## YOLO

In [1]:
import csv
import re
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
from paddleocr import PaddleOCR
from ultralytics import YOLO

c:\Users\alici\anaconda3\envs\VC_P4\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### PaddleOCR

In [2]:
VIDEO_PATH = "C0142.mp4"
OUTPUT_VIDEO = "resultado.mp4"
OUTPUT_CSV = "detecciones.csv"

# Modelos
model_general = YOLO("yolo11n.pt")                                              # personas y vehículos (COCO)
model_plates = YOLO("runs/matriculas_yolo11n_1280_min001/weights/best.pt")       # matrículas (nuestro modelo)

# OCR
reader = PaddleOCR(
    lang="en",
    device="gpu",
    ocr_version="PP-OCRv5",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

# Personas, bicicletas y vehículos
CLASSES = [0, 1, 2, 3, 5, 7]

# Clases de vehículos con matrícula: coche, moto, autobús, camión
VEHICLE_CLASSES = [2, 3, 5, 7]

Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alici\.paddlex\official_models\PP-OCRv5_server_det`.
c:\Users\alici\anaconda3\envs\VC_P4\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('en_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\alici\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


In [ ]:
# Lee el texto de una matrícula con PaddleOCR. Devuelve (texto, confianza)
def read_plate(crop):
    if crop is None or crop.size == 0:
        return "", 0.0

    # Ampliar matrícula
    crop = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

    # Convertir BGR a RGB
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

    # Ejecutar PaddleOCR
    results = reader.predict(crop_rgb)

    if not results:
        return "", 0.0

    candidates = []

    for result in results:

        data = result.json

        if isinstance(data, dict):
            data = data.get("res", data)

        texts = data.get("rec_texts", [])
        scores = data.get("rec_scores", [])

        for text, confidence in zip(texts, scores):

            text = re.sub(r"[^A-Z0-9]", "", text.upper())

            confidence = float(confidence)

            if re.fullmatch(r"(?:[0-9]{4}[A-Z]{3}|[A-Z]{2}[0-9]{3}[A-Z]{2})", text):
                candidates.append((text, confidence))

    if not candidates:
        return "", 0.0

    # Seleccionar lectura de mayor confianza
    return max(candidates, key=lambda x: x[1])


# Guarda una lectura del vehículo y actualiza su matrícula más repetida
def update_ocr(vehicle_id, text, confidence):
    if vehicle_id is None or not text:
        return

    # Evitar lecturas de muy baja confianza
    if confidence < 0.30:
        return

    ocr_history[vehicle_id].append((text, confidence))

    # Contabilizar lecturas
    counts = Counter(item[0] for item in ocr_history[vehicle_id])

    # Elegir la lectura más repetida
    selected_text, occurrences = counts.most_common(1)[0]

    # Confianza media de las lecturas seleccionadas
    selected_confidences = [conf for plate, conf in ocr_history[vehicle_id] if plate == selected_text]

    best_plates[vehicle_id] = {
        "text": selected_text,
        "confidence": float(np.mean(selected_confidences)),
        "occurrences": occurrences
    }


# Asigna cada matrícula al vehículo MÁS PEQUEÑO cuya caja contiene su centro
def asignar_matriculas(boxes, plates):
    plate_of_vehicle = {}

    vehicles = [(i, *map(int, b.xyxy[0].tolist()))
                for i, b in enumerate(boxes) if int(b.cls[0]) in VEHICLE_CLASSES]

    for plate in plates:
        cx = (plate["x1"] + plate["x2"]) / 2
        cy = (plate["y1"] + plate["y2"]) / 2

        # Vehículos que contienen el centro de la matrícula, con su área
        inside = [(i, (vx2 - vx1) * (vy2 - vy1)) for i, vx1, vy1, vx2, vy2 in vehicles
                  if vx1 <= cx <= vx2 and vy1 <= cy <= vy2]

        if inside:
            i_min = min(inside, key=lambda v: v[1])[0]      # el más pequeño
            current = plate_of_vehicle.get(i_min)
            # Si ese vehículo ya tenía otra matrícula, se queda la de mayor confianza
            if current is None or plate["conf"] > current["conf"]:
                plate_of_vehicle[i_min] = plate

    return plate_of_vehicle

In [12]:
# Lecturas de cada vehículo
ocr_history = defaultdict(list)
best_plates = {}

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

# Cabecera csv
headers = [
    "fotograma",
    "tipo_objeto",
    "confianza",
    "identificador_tracking",
    "x1", "y1", "x2", "y2",
    "matricula_en_su_caso",
    "confianza_matricula",
    "mx1", "my1", "mx2", "my2",
    "texto_matricula",
    "confianza_ocr",
    "lecturas_coincidentes"
]

csv_file = open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig")
writer = csv.DictWriter(csv_file, fieldnames=headers)
writer.writeheader()

frame_number = 0

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # Detección y seguimiento de personas y vehículos
    result_general = model_general.track(
        frame,
        persist=True,
        tracker="custom_botsort.yaml",
        classes=CLASSES,
        conf=0.1,
        verbose=False
    )[0]

    # Detección de matrículas (a 1280, la resolución con la que se entrenó el modelo)
    result_plates = model_plates.predict(
        frame,
        conf=0.4,
        imgsz=1280,
        verbose=False
    )[0]

    plates = []

    if result_plates.boxes is not None:

        for box in result_plates.boxes:

            px1, py1, px2, py2 = map(int, box.xyxy[0].tolist())

            px1 = max(0, px1)
            py1 = max(0, py1)
            px2 = min(width, px2)
            py2 = min(height, py2)

            if px2 <= px1 or py2 <= py1:
                continue

            plate_conf = float(box.conf[0])

            plate_crop = frame[py1:py2, px1:px2]

            # OCR
            plate_text, ocr_conf = read_plate(plate_crop)

            plates.append({
                "x1": px1,
                "y1": py1,
                "x2": px2,
                "y2": py2,
                "conf": plate_conf,
                "text": plate_text,
                "ocr_conf": ocr_conf
            })

    boxes = result_general.boxes

    if boxes is not None:

        # Matrícula que corresponde a cada vehículo de este fotograma
        plate_of_vehicle = asignar_matriculas(boxes, plates)

        for i, box in enumerate(boxes):

            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            confidence = float(box.conf[0])
            class_id = int(box.cls[0])

            object_type = model_general.names[class_id]

            tracking_id = None

            if boxes.id is not None:
                tracking_id = int(boxes.id[i])

            matched_plate = plate_of_vehicle.get(i)

            if matched_plate and tracking_id is not None:

                update_ocr(
                    tracking_id,
                    matched_plate["text"],
                    matched_plate["ocr_conf"]
                )

            # Matrícula más fiable acumulada
            plate_info = best_plates.get(tracking_id, {}) if tracking_id is not None else {}

            final_text = plate_info.get("text", "")
            final_ocr_conf = plate_info.get("confidence", "")
            occurrences = plate_info.get("occurrences", 0)

            # Guarda la fila en el csv
            row = {
                "fotograma": frame_number,
                "tipo_objeto": object_type,
                "confianza": confidence,
                "identificador_tracking": tracking_id,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "matricula_en_su_caso": int(matched_plate is not None),
                "confianza_matricula": matched_plate["conf"] if matched_plate else "",
                "mx1": matched_plate["x1"] if matched_plate else "",
                "my1": matched_plate["y1"] if matched_plate else "",
                "mx2": matched_plate["x2"] if matched_plate else "",
                "my2": matched_plate["y2"] if matched_plate else "",
                "texto_matricula": final_text,
                "confianza_ocr": final_ocr_conf,
                "lecturas_coincidentes": occurrences
            }

            writer.writerow(row)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            label = f"{object_type} ID:{tracking_id}"

            if final_text:
                label += f" | {final_text}"

            cv2.putText(frame, label, (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    for plate in plates:

        cv2.rectangle(frame, (plate["x1"], plate["y1"]), (plate["x2"], plate["y2"]), (0, 0, 255), 2)

        label = plate["text"] if plate["text"] else "Matricula"

        cv2.putText(frame, label, (plate["x1"], max(20, plate["y1"] - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Guarda el vídeo
    out.write(frame)

    if frame_number % 100 == 0:
        print(f"Procesados {frame_number} fotogramas")

cap.release()
out.release()
csv_file.close()

print("Procesamiento finalizado")
print("Vídeo:", OUTPUT_VIDEO)
print("CSV:", OUTPUT_CSV)

Procesados 100 fotogramas
Procesados 200 fotogramas
Procesados 300 fotogramas
Procesados 400 fotogramas
Procesados 500 fotogramas
Procesados 600 fotogramas
Procesados 700 fotogramas
Procesados 800 fotogramas
Procesados 900 fotogramas
Procesados 1000 fotogramas
Procesados 1100 fotogramas
Procesados 1200 fotogramas
Procesados 1300 fotogramas
Procesados 1400 fotogramas
Procesados 1500 fotogramas
Procesados 1600 fotogramas
Procesados 1700 fotogramas
Procesados 1800 fotogramas
Procesados 1900 fotogramas
Procesados 2000 fotogramas
Procesados 2100 fotogramas
Procesados 2200 fotogramas
Procesados 2300 fotogramas
Procesados 2400 fotogramas
Procesados 2500 fotogramas
Procesados 2600 fotogramas
Procesados 2700 fotogramas
Procesados 2800 fotogramas
Procesamiento finalizado
Vídeo: resultado.mp4
CSV: detecciones.csv


In [13]:
df = pd.read_csv("detecciones.csv")

conteo_unicos = (
    df[df["identificador_tracking"].notna()]
    .groupby("tipo_objeto")["identificador_tracking"]
    .nunique()
)

print("Total de objetos únicos:")
print(conteo_unicos)

Total de objetos únicos:
tipo_objeto
bicycle        2
bus            2
car           64
motorcycle     6
person        20
truck          3
Name: identificador_tracking, dtype: int64


### Extra: Anonimizar personas y vehículos

In [2]:

def anonimizar_detecciones(frame, detecciones):

    for deteccion in detecciones:

        x1 = deteccion["x1"]
        y1 = deteccion["y1"]
        x2 = deteccion["x2"]
        y2 = deteccion["y2"]

        # Recortar región detectada
        roi = frame[y1:y2, x1:x2]

        if roi.size == 0:
            continue

        h, w = roi.shape[:2]

        # Pixelado intenso
        reducido = cv2.resize(
            roi,
            (max(1, w // 15), max(1, h // 15)),
            interpolation=cv2.INTER_LINEAR
        )

        pixelado = cv2.resize(
            reducido,
            (w, h),
            interpolation=cv2.INTER_NEAREST
        )

        # Aplicar anonimización
        frame[y1:y2, x1:x2] = pixelado

    return frame


In [ ]:
VIDEO_PATH = "C0142.mp4"
OUTPUT_VIDEO = "resultado_pixelado.mp4"

# Personas, bicicletas y vehículos
CLASSES = [0, 1, 2, 3, 5, 7]

# Modelos
model_general = YOLO("yolo11n.pt")                                              # personas y vehículos (COCO)
model_plates = YOLO("runs/matriculas_yolo11n_1280_min001/weights/best.pt")       # matrículas (nuestro modelo)

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

frame_number = 0

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_number += 1

    # Detección y seguimiento de personas y vehículos
    result_general = model_general.track(
        frame,
        persist=True,
        tracker="custom_bytetrack.yaml",
        classes=CLASSES,
        conf=0.5,
        verbose=False
    )[0]

    # Detección de matrículas (a 1280, la resolución con la que se entrenó el modelo)
    result_plates = model_plates.predict(
        frame,
        conf=0.4,
        imgsz=1280,
        verbose=False
    )[0]

    detecciones = []

    if result_plates.boxes is not None:

        for box in result_plates.boxes:

            px1, py1, px2, py2 = map(int, box.xyxy[0].tolist())

            px1 = max(0, px1)
            py1 = max(0, py1)
            px2 = min(width, px2)
            py2 = min(height, py2)

            if px2 <= px1 or py2 <= py1:
                continue


            detecciones.append({
                "x1": px1,
                "y1": py1,
                "x2": px2,
                "y2": py2,
                "tipo": "matricula"
            })

    boxes = result_general.boxes

    if boxes is not None:

        # Matrícula que corresponde a cada vehículo de este fotograma

        for i, box in enumerate(boxes):

            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            confidence = float(box.conf[0])
            class_id = int(box.cls[0])

            if class_id == 0:
                detecciones.append({
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "tipo": "persona"
                })

    frame = anonimizar_detecciones(frame, detecciones);

    # Guarda el vídeo
    out.write(frame)

    if frame_number % 100 == 0:
        print(f"Procesados {frame_number} fotogramas")

cap.release()
out.release()

print("Procesamiento finalizado")
print("Vídeo:", OUTPUT_VIDEO)

Procesados 100 fotogramas
Procesados 200 fotogramas
Procesados 300 fotogramas
Procesados 400 fotogramas
Procesados 500 fotogramas
Procesados 600 fotogramas
Procesados 700 fotogramas
Procesados 800 fotogramas
Procesados 900 fotogramas
Procesados 1000 fotogramas
Procesados 1100 fotogramas
Procesados 1200 fotogramas
Procesados 1300 fotogramas
